# Laplace Approximation: Logistic Regression

This example opens the *Universal Inference* section. The idea behind the section is simple: if every continuous latent variable is approximated by a Gaussian, any factor that has a log-density can send a message, so inference can run in virtually any model, at the cost of an approximation. The classic way to fit such a Gaussian is the **Laplace approximation**: find the mode of the local log-density and use the curvature at the mode as the precision.

In RxInfer this takes one keyword, `fallback = LocalLaplace()`. Wherever a model has exact message update rules, RxInfer uses them; wherever it has none, the fallback steps in:

1. the **rule fallback** computes the missing messages, marginals and average energies from the nodes' log-densities, as expectations over the other variables' beliefs;
2. a **form constraint** projects every belief that has no closed form onto a Gaussian by the Laplace method, in the unconstrained coordinates of its domain, and normalises the beliefs of discrete variables exactly;
3. the **expectations** are computed by the backends of the loaded packages, Gauss–Hermite quadrature over Gaussians or an exact sum over a categorical, chosen once for each call of `infer`;
4. every variable is **initialised** from its prior, where an iterative scheme needs a start.

Anything you set yourself, a rule fallback, a form constraint or an initialisation, wins over the keyword.

We test it on the textbook bioassay experiment from *Bayesian Data Analysis* (Gelman et al., 3rd ed., Section 3.7; originally Racine-Poon et al., 1986), where the exact posterior can be computed on a grid, so we can see exactly what the Laplace approximation gets right and what it misses.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra
using SpecialFunctions, StatsFuns, StableRNGs, Plots, PrettyTables
import ExpectationApproximations as EA

## The bioassay experiment

Four groups of five animals received a dose $x_i$ (in log g/ml) of a compound, and $y_i$ animals died in group $i$:

In [ ]:
x = [-0.86, -0.30, -0.05, 0.73]
n = [5, 5, 5, 5]
y = [0, 1, 3, 5]

The model is a logistic regression with a Gaussian prior on $\theta = (\alpha, \beta)$:

$$\begin{aligned}
\theta &\sim \mathcal{N}\left(0, \mathrm{diag}(4^2, 20^2)\right),\\
y_i &\sim \mathrm{Binomial}\left(n_i, \sigma(\alpha + \beta x_i)\right),
\end{aligned}$$

where $\sigma$ is the logistic function. The pair $\mathrm{Binomial}$ and $\sigma$ has no conjugate update rules. We write the likelihood as a single stochastic node, `BinomialLogistic`: a small distribution type whose only job is to evaluate the log-likelihood. No message update rule is written for it. The node has the interfaces `out` (the count), `θ`, and the constants `x` and `n`.

In [ ]:
logbinomial(n, k) = loggamma(n + 1) - loggamma(k + 1) - loggamma(n - k + 1)

struct BinomialLogistic{T, X, N} <: DiscreteUnivariateDistribution
    θ::T
    x::X
    n::N
end

function BayesBase.logpdf(d::BinomialLogistic, y)
    η = d.θ[1] + d.θ[2] * d.x
    return -y * log1pexp(-η) - (d.n - y) * log1pexp(η) + logbinomial(d.n, y)
end
BayesBase.insupport(::BinomialLogistic, y) = true

@define_factor_node(node = BinomialLogistic, type = Stochastic, interfaces = [:out, :θ, :x, :n])

In [ ]:
prior_θ = MvNormalMeanCovariance(zeros(2), diagm([4.0^2, 20.0^2]))

@model function bioassay(y, x, n, prior)
    θ ~ prior
    for i in eachindex(y)
        y[i] ~ BinomialLogistic(θ, x[i], n[i])
    end
end

The inference call has one addition to a plain `infer` call, the keyword `fallback`. Nothing else is model-specific: no rules, no constraints, no initialisation, no iterations. The posterior of $\theta$ is the Gaussian prior times four messages from the likelihood nodes, which are log-densities with no closed form, and the Laplace projection turns that product into a Gaussian.

We also ask for the free energy and for the log-normalisers of the beliefs, `logscales = true`, which the section on the evidence reads, and for a report of what the keyword approximated, `report = true`. The posterior then comes back as RxInfer's marginal, which keeps its log-normaliser and otherwise behaves as its distribution: its moments, its density and its samples are the Gaussian's.

In [ ]:
result = infer(
    model = bioassay(x = x, n = n, prior = prior_θ),
    data = (y = y,),
    fallback = LocalLaplace(),
    free_energy = true,
    logscales = true,
    report = true,
)

q_θ = result.posteriors[:θ]

As a sanity check, we apply the Laplace method directly to the unnormalised log-posterior, written by hand, with `laplace` from ExpectationApproximations.jl, the package behind the fallback. Message passing must assemble exactly the same density, so both Gaussians must agree to the tolerance of the Newton search.

In [ ]:
logposterior(θ) = logpdf(prior_θ, θ) + sum(logpdf(BinomialLogistic(θ, x[i], n[i]), y[i]) for i in eachindex(y))

q_direct = EA.laplace(logposterior, zeros(2))

println("max |mean difference|       = ", maximum(abs, mean(q_θ) - mean(q_direct.distribution)))
println("max |covariance difference| = ", maximum(abs, cov(q_θ) - cov(q_direct.distribution)))
@assert isapprox(mean(q_θ), mean(q_direct.distribution); rtol = 1e-8) # hide
@assert isapprox(cov(q_θ), cov(q_direct.distribution); rtol = 1e-8) # hide

## What was approximated

The report lists each approximated posterior: how it was projected, and which nodes' messages formed it, by an exact rule or by the rule fallback. It warns about the parts most likely to be inaccurate. With PrettyTables.jl loaded, it prints as tables.

In [ ]:
@assert isempty(RxInfer.diagnostics(result.report)) # hide
@assert RxInfer.node_types(result.report.approximations)[(BinomialLogistic, :message, :θ)].fallbacks == length(y) # hide
@assert result.report.approximations.variables[(:θ, nothing)].marginal.outcome.converged # hide
result.report.approximations

The posterior of $\theta$ is the prior's message, from its exact rule, times the four likelihood messages from the rule fallback. Their expectations are exact, because the likelihood's other inputs are data and constants. Laplace's method converged, and the normaliser it found is the evidence of the section below. The report raises no diagnostic. Its other section, `result.report.timings`, says where the time went.

## Exact posterior on a grid

With two parameters, the exact posterior can be evaluated on a fine grid. This is our ground truth.

In [ ]:
αs = range(-4, 6; length = 400)
βs = range(-10, 40; length = 400)
logp_grid = [logposterior([α, β]) for α in αs, β in βs]
p_grid = exp.(logp_grid .- maximum(logp_grid))
p_grid ./= sum(p_grid)

A = [α for α in αs, β in βs]
B = [β for α in αs, β in βs]
grid_mean = [sum(p_grid .* A), sum(p_grid .* B)]
grid_cov = [sum(p_grid .* (A .- grid_mean[1]) .* (A .- grid_mean[1])) sum(p_grid .* (A .- grid_mean[1]) .* (B .- grid_mean[2]));
            sum(p_grid .* (A .- grid_mean[1]) .* (B .- grid_mean[2])) sum(p_grid .* (B .- grid_mean[2]) .* (B .- grid_mean[2]))]

summary_row(name, m, C) = println(rpad(name, 10),
    "E[α] = ", lpad(round(m[1]; digits = 2), 6), "   sd(α) = ", lpad(round(sqrt(C[1, 1]); digits = 2), 5),
    "   E[β] = ", lpad(round(m[2]; digits = 2), 6), "   sd(β) = ", lpad(round(sqrt(C[2, 2]); digits = 2), 5),
    "   corr = ", round(C[1, 2] / sqrt(C[1, 1] * C[2, 2]); digits = 2))

summary_row("grid", grid_mean, grid_cov)
summary_row("Laplace", mean(q_θ), cov(q_θ))
@assert 0.6 < mean(q_θ)[2] / grid_mean[2] < 0.8 # hide
@assert sqrt(cov(q_θ)[2, 2]) < sqrt(grid_cov[2, 2]) # hide

The Laplace approximation is centred at the mode, while the exact posterior of $\beta$ is skewed to the right, so its mean lies above the mode, and its spread is larger than the curvature at the mode suggests. The contour plot shows the same picture: the Laplace ellipse (dashed) matches the core of the posterior, but not the long tail towards large $\beta$. Both densities are drawn relative to their maximum, at the same levels.

In [ ]:
p_laplace = [pdf(q_θ, [α, β]) for α in αs, β in βs]

# Both densities relative to their maximum, drawn at the same levels
levels = [0.05, 0.2, 0.4, 0.6, 0.8]
plt = contour(αs, βs, (p_grid ./ maximum(p_grid))'; levels = levels, seriescolor = :blue, colorbar = false, linewidth = 2,
    xlabel = "α", ylabel = "β", title = "Bioassay posterior", xlims = (-2, 5), ylims = (-5, 30))
contour!(plt, αs, βs, (p_laplace ./ maximum(p_laplace))'; levels = levels, seriescolor = :red, linestyle = :dash, colorbar = false, linewidth = 2)
plot!(plt, [NaN], [NaN]; color = :blue, linewidth = 2, label = "exact (grid)")
plot!(plt, [NaN], [NaN]; color = :red, linestyle = :dash, linewidth = 2, label = "Laplace")
plt

## The evidence and the free energy

The grid also gives the exact log-evidence $\log p(y)$, the logarithm of the normaliser of the posterior. Message passing gives two approximations of it:

- the **log-normaliser of the posterior**, which `infer` kept with `logscales = true`. Every message from a likelihood node is exact here, so it is Laplace's approximation of the evidence, $\log p(y) \approx \ell(\hat{\theta}) + \frac{d}{2}\log 2\pi - \frac{1}{2}\log\det\left(-\nabla^2 \ell(\hat{\theta})\right)$;
- the **free energy** $F[q] = -\log p(y) + \mathrm{KL}\left(q(\theta) \,\|\, p(\theta \mid y)\right)$, an upper bound on $-\log p(y)$, which it misses by the divergence of the Gaussian from the exact posterior.

In [ ]:
logevidence_laplace = getlogscale(q_θ)
free_energy = only(result.free_energy)

cell_area = step(αs) * step(βs)
logevidence_grid = maximum(logp_grid) + log(sum(exp.(logp_grid .- maximum(logp_grid))) * cell_area)

println("-log p(y):  grid = ", round(-logevidence_grid; digits = 3), ",  Laplace = ", round(-logevidence_laplace; digits = 3), ",  free energy = ", round(free_energy; digits = 3))
println("KL(q || exact posterior) = ", round(free_energy + logevidence_grid; digits = 3))
@assert isapprox(logevidence_laplace, q_direct.logZ; rtol = 1e-8) # hide
@assert free_energy > -logevidence_grid # hide
@assert abs(logevidence_laplace - logevidence_grid) < 0.1 # hide

Laplace's evidence is close to the exact one, while the free energy is about one nat above it: the Gaussian misses the skew of the posterior, and the free energy measures by how much.

## CVI: the best Gaussian

Laplace's Gaussian is centred at the mode of the posterior, with its curvature there. It is one Gaussian among many; the best one, in the sense of variational inference, minimises $\mathrm{KL}(q \,\|\, p(\theta \mid y))$, or maximises the evidence lower bound. `fallback = CVI()` keeps the same rules and projects onto that Gaussian instead: the fixed point of conjugate-computation variational inference. With one latent variable the free energy is the negative bound, so CVI's is the lowest free energy any Gaussian reaches.

In [ ]:
result_cvi = infer(
    model = bioassay(x = x, n = n, prior = prior_θ),
    data = (y = y,),
    fallback = CVI(),
    free_energy = true,
)
q_cvi = result_cvi.posteriors[:θ]

summary_row("grid", grid_mean, grid_cov)
summary_row("Laplace", mean(q_θ), cov(q_θ))
summary_row("CVI", mean(q_cvi), cov(q_cvi))
println("-log p(y) = ", round(-logevidence_grid; digits = 3), ";  free energy: Laplace = ", round(free_energy; digits = 3), ", CVI = ", round(only(result_cvi.free_energy); digits = 3))
@assert -logevidence_grid < only(result_cvi.free_energy) < free_energy # hide
@assert abs(mean(q_cvi)[2] - grid_mean[2]) < abs(mean(q_θ)[2] - grid_mean[2]) # hide
@assert sqrt(cov(q_cvi)[2, 2]) < sqrt(grid_cov[2, 2]) # hide

The variational Gaussian moves towards the heavier tail of the posterior of $\beta$: its mean is close to the exact one, where Laplace's is about a third low, and its free energy is within a fraction of a nat of $-\log p(y)$. Its spread is smaller than the exact one, the known price of $\mathrm{KL}(q \,\|\, p)$, which penalises mass where the posterior has little. It is still a symmetric Gaussian, so the skew itself remains out of reach.

## A derived quantity: the LD50

The quantity of interest in the original study is the LD50, the dose at which the probability of death is 50%, $\mathrm{LD50} = -\alpha / \beta$. It is a nonlinear function of $\theta$, so we compare its distribution under the exact posterior (sampled from the grid) and under the Laplace approximation, keeping only samples with $\beta > 0$ as in the book.

In [ ]:
rng = StableRNG(42)
nsamples = 10_000
idx = rand(rng, Categorical(vec(p_grid)), nsamples)
ld50_grid = [-A[i] / B[i] for i in idx if B[i] > 0]
ld50_laplace = [-θ[1] / θ[2] for θ in eachcol(rand(rng, q_θ, nsamples)) if θ[2] > 0]

histogram(ld50_grid; bins = range(-0.6, 0.6; length = 61), normalize = :pdf, alpha = 0.5, label = "exact (grid)", xlabel = "LD50", title = "Posterior of the LD50")
histogram!(ld50_laplace; bins = range(-0.6, 0.6; length = 61), normalize = :pdf, alpha = 0.5, label = "Laplace")

In [ ]:
p_positive_laplace = 1 - cdf(Normal(mean(q_θ)[2], sqrt(cov(q_θ)[2, 2])), 0)
interval_grid, interval_laplace = quantile(ld50_grid, [0.05, 0.95]), quantile(ld50_laplace, [0.05, 0.95])
println("P(β > 0):  grid = ", round(sum(p_grid[B .> 0]); digits = 4), ",  Laplace = ", round(p_positive_laplace; digits = 4))
println("LD50 median:  grid = ", round(median(ld50_grid); digits = 3), ",  Laplace = ", round(median(ld50_laplace); digits = 3))
println("LD50 90% interval:  grid = ", round.(interval_grid; digits = 3), ",  Laplace = ", round.(interval_laplace; digits = 3))
@assert 0.01 < 1 - p_positive_laplace < 0.1 # hide
@assert abs(median(ld50_grid) - median(ld50_laplace)) < 0.05 # hide
@assert only(diff(interval_laplace)) > only(diff(interval_grid)) # hide

The medians agree, but the Laplace interval for the LD50 is wider than the exact one. The Gaussian is symmetric around the mode of $\beta$, so it puts a few percent of its mass near and below $\beta = 0$, a region the data rule out, and dividing by small values of $\beta$ spreads the LD50 out.

## The same keyword, another model: Poisson regression

Nothing in the fallback knows about logistic regression. To show that, we fit a Poisson regression with a log link, $y_i \sim \mathrm{Poisson}(\exp(a + b t_i))$, on simulated counts. The only new code is the likelihood node.

In [ ]:
struct PoissonLogLinear{T, X} <: DiscreteUnivariateDistribution
    θ::T
    t::X
end

BayesBase.logpdf(d::PoissonLogLinear, y) = (η = d.θ[1] + d.θ[2] * d.t; y * η - exp(η) - loggamma(y + 1))
BayesBase.insupport(::PoissonLogLinear, y) = true

@define_factor_node(node = PoissonLogLinear, type = Stochastic, interfaces = [:out, :θ, :t])

@model function poisson_regression(y, t, prior)
    θ ~ prior
    for i in eachindex(y)
        y[i] ~ PoissonLogLinear(θ, t[i])
    end
end

θ_true = [0.5, 1.2]
t = collect(range(-1, 1.5; length = 30))
counts = [rand(rng, Poisson(exp(θ_true[1] + θ_true[2] * ti))) for ti in t]

result_poisson = infer(
    model = poisson_regression(t = t, prior = MvNormalMeanCovariance(zeros(2), diagm([10.0, 10.0]))),
    data = (y = counts,),
    fallback = LocalLaplace(),
)

q_poisson = result_poisson.posteriors[:θ]
println("true θ = ", θ_true, ",  posterior mean = ", round.(mean(q_poisson); digits = 3), ",  posterior sd = ", round.(sqrt.(diag(cov(q_poisson))); digits = 3))
@assert all(abs.(mean(q_poisson) .- θ_true) .< 2 .* sqrt.(diag(cov(q_poisson)))) # hide

In [ ]:
ts = range(-1, 1.5; length = 100)
draws = rand(rng, q_poisson, 200)
rates = [exp(θ[1] + θ[2] * τ) for τ in ts, θ in eachcol(draws)]
lower, upper = [quantile(r, 0.05) for r in eachrow(rates)], [quantile(r, 0.95) for r in eachrow(rates)]
middle = [exp(mean(q_poisson)[1] + mean(q_poisson)[2] * τ) for τ in ts]

plot(ts, middle; ribbon = (middle .- lower, upper .- middle), fillalpha = 0.3, label = "posterior rate (90%)", xlabel = "t", ylabel = "count")
plot!(ts, exp.(θ_true[1] .+ θ_true[2] .* ts); color = :black, linestyle = :dash, label = "true rate")
scatter!(t, counts; label = "observed counts", color = :gray)

## The model as it reads on paper

The likelihood node fuses the logistic function and the binomial. The model can also be written as it reads on paper, with the probability of death a deterministic function of $\theta$, $p = \sigma(\alpha + \beta x)$. RxInfer has no binomial node, so each group of five animals becomes five Bernoulli trials, which give the same posterior: the binomial coefficient does not depend on $\theta$.

Two deterministic nodes now stand between $\theta$ and the data: RxInfer's own `dot` node for the linear predictor, and a Delta node for `logistic`, a function no package declares as a node. Both go through the same rule fallback, which pulls the message from the observation back through their functions. `dot` keeps its exact rules wherever its messages are Gaussian, as on the way from $\theta$ towards the data. The posterior must be the one the fused node gave.

In [ ]:
doses = reduce(vcat, [fill(x[i], n[i]) for i in eachindex(x)])
deaths = reduce(vcat, [[ones(y[i]); zeros(n[i] - y[i])] for i in eachindex(x)])

@model function bioassay_trials(y, doses, prior)
    θ ~ prior
    for j in eachindex(y)
        p[j] := logistic(dot([1.0, doses[j]], θ))
        y[j] ~ Bernoulli(p[j])
    end
end

result_trials = infer(
    model = bioassay_trials(doses = doses, prior = prior_θ),
    data = (y = deaths,),
    fallback = LocalLaplace(),
    report = true,
)

q_trials = result_trials.posteriors[:θ]
println("max |mean difference to the fused node|       = ", maximum(abs, mean(q_trials) - mean(q_θ)))
println("max |covariance difference to the fused node| = ", maximum(abs, cov(q_trials) - cov(q_θ)))
@assert isapprox(mean(q_trials), mean(q_θ); rtol = 1e-8) # hide
@assert isapprox(cov(q_trials), cov(q_θ); rtol = 1e-8) # hide

The report of this form shows the two deterministic nodes at work.
- Towards its output, `dot` runs its exact rule: its input there is the Gaussian message of $\theta$.
- Towards $\theta$, the rule fallback runs. The message there comes from `logistic`, and it is a log-density.

The note says why that message was not projected: on its own, the likelihood of one Bernoulli observation has no mode.

In [ ]:
@assert [d.kind for d in RxInfer.diagnostics(result_trials.report)] == [:unprojected_messages] # hide
@assert RxInfer.node_types(result_trials.report.approximations)[(dot, :message, :in2)].fallbacks == length(deaths) # hide
@assert RxInfer.node_types(result_trials.report.approximations)[(dot, :message, :out)].rules >= length(deaths) # hide
result_trials.report.approximations

This form has no free energy yet. The message from the Delta node towards $p$ is a Gaussian on the real line, which ignores that a probability lies between $0$ and $1$, and the Bernoulli node's average energy needs $\mathbb{E}[\log p]$.

## Exact vs approximate, and what is still missing

- **What worked.** One keyword ran a textbook Laplace approximation in two non-conjugate models, without a single hand-written message update rule, constraint or initialisation, and `report = true` said what it approximated. The result coincides with a Laplace approximation of the hand-written posterior to numerical precision, and the log-normaliser of the posterior is Laplace's evidence.
- **Where the approximation shows.** The Laplace approximation is centred at the mode and uses local curvature, so it misses the skew of the bioassay posterior in $\beta$: the posterior mean of $\beta$ is underestimated by about a third, and the free energy sits about one nat above the exact $-\log p(y)$. Because a Gaussian is symmetric, it also puts a few percent of its mass on $\beta < 0$, which the data rule out, and this inflates the uncertainty of the LD50. Moment matching (expectation propagation) is the natural next step to capture the mean and spread better; it is on the roadmap for this section.
- **What is still missing.**
  - The model written with deterministic nodes runs, but has no free energy: the Gaussian belief of `p` ignores that a probability lies between $0$ and $1$. RxInfer has no `Binomial` node either, so the trials are observed one by one.
  - $\theta$ is a single two-dimensional variable. With separate scalar variables $\alpha$ and $\beta$, the graph would have loops, and the messages between them would not carry their strong posterior correlation.